# LoRA on a free T4 — Qwen2.5-1.5B for dialogue summarization

**Runtime → Change runtime type → T4 GPU** before running anything.

### Why LoRA here, and why not on an encoder

LoRA earns its place when full fine-tuning is not an option. RoBERTa-base is 125M parameters — it fully fine-tunes on this same T4 in about five minutes using under 4GB. Putting LoRA on it trades a little accuracy for nothing in return.

A 1.5B decoder is a different situation. AdamW keeps two fp32 moments per *trainable* parameter, so full fine-tuning needs roughly 24GB before activations are counted. The T4 has 15GB.

| Approach | GPU memory | On a T4 |
|---|---|---|
| Full fine-tune (fp16 weights, AdamW) | ~24 GB | does not fit |
| **LoRA, fp16 base** | **~7 GB** | **comfortable** |
| QLoRA, 4-bit base | ~4 GB | only needed for larger models |

Freezing the base and training a low-rank update removes the optimizer-state term almost entirely, which is where the memory actually goes. Quantization is a further step down, and this model does not need it — so the base stays in fp16 and nothing is lost to 4-bit rounding.

### What it measures

A fine-tuned model is only interesting relative to what you could have had for free. Two baselines run *before* training:

1. **Lead-3** — just take the first three turns. The extractive baseline that a surprising number of summarization papers fail to beat.
2. **Zero-shot base model** — the same Qwen2.5-1.5B-Instruct, same prompt, no adapter. This is the one that matters: if LoRA does not beat it, the training run bought you nothing.

### Runtime on a free T4

Roughly 40-50 minutes end to end: ~3 min setup, ~6 min baselines, ~25 min training, ~10 min evaluation generation.


In [1]:
# ============================================================
# 1. Dependencies
# ============================================================
!pip install -q -U "transformers>=4.44" "datasets>=2.19" accelerate peft
!pip install -q rouge_score huggingface_hub matplotlib

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 19.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
  Preparing metadata (setup.py) ... done
torch: 2.11.0+cu130 | cuda: True
Tesla T4, 15360 MiB, 7.5


In [3]:
# ============================================================
# 2. Imports, configuration, seeding
# ============================================================
import gc, inspect, json, math, os, random, re, textwrap, time
from datetime import date

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed,
)
from peft import LoraConfig, get_peft_model, PeftModel
from rouge_score import rouge_scorer

# ---- Model ----------------------------------------------------
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"   # not gated, fits a T4 in fp16

# ---- LoRA -----------------------------------------------------
LORA_R        = 16       # rank of the update. 8 is cheaper, 32 holds more capacity.
LORA_ALPHA    = 32       # scaling = alpha / r = 2.0
LORA_DROPOUT  = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]   # attention + MLP

# ---- Training -------------------------------------------------
LEARNING_RATE = 2e-4     # LoRA territory. 2e-5 would barely move the adapters.
EPOCHS        = 2
MICRO_BATCH   = 2        # what fits on a T4 at this sequence length
GRAD_ACCUM    = 8        # effective batch = 16
MAX_LENGTH    = 768
WARMUP_RATIO  = 0.05
WEIGHT_DECAY  = 0.0      # the adapter is already a tiny, constrained parameter set
SEED          = 42

# ---- Budget (lower these if Colab disconnects) ----------------
MAX_TRAIN_SAMPLES = 6000   # None = all ~12.4k
EVAL_SAMPLES      = 250    # generation is the slow part of evaluation
GEN_BATCH         = 8
MAX_NEW_TOKENS    = 96

# ---- Output / Hub ---------------------------------------------
OUTPUT_DIR   = "/content/qwen-dialogsum"
ADAPTER_DIR  = os.path.join(OUTPUT_DIR, "adapter")
HF_USERNAME  = "emonHasan01"            # <-- change this
REPO_NAME    = "QwenSum-LoRA"
REPO_ID      = f"{HF_USERNAME}/{REPO_NAME}"
PUSH_TO_HUB  = True
PUSH_MERGED  = False      # also push a merged fp16 copy (~3GB upload)

set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(ADAPTER_DIR, exist_ok=True)

# T4 is Turing: fp16 yes, bf16 no, FlashAttention-2 no (needs Ampere+).
CAP = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
SUPPORTS_BF16 = CAP[0] >= 8
DTYPE     = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"
print(f"compute capability {CAP} | bf16 {SUPPORTS_BF16} | dtype {DTYPE} | attn {ATTN_IMPL}")

RESULTS = {}   # every measured number lands here and feeds the model card

compute capability (7, 5) | bf16 False | dtype torch.float16 | attn sdpa


## 1. Task and data

**DialogSum** — 12.5k everyday conversations with human-written abstractive summaries. Chosen deliberately over a news corpus:

- Short inputs, so a 768-token window is generous rather than a truncation problem
- Abstractive by construction — the summary rewrites rather than copies, so the extractive baseline has a real ceiling
- Permissively licensed and small enough to train inside a free Colab session

The loader normalises whatever mirror responds down to two columns, `dialogue` and `summary`, so a schema change upstream does not silently produce an empty field.

In [4]:
# ============================================================
# 3. Load and normalise the dataset
# ============================================================
CANDIDATES = [
    ("knkarthick/dialogsum", "dialogue", "summary"),
    ("neil-code/dialogsum-test", "dialogue", "summary"),
    ("knkarthick/samsum", "dialogue", "summary"),
]

raw, DATASET_REPO = None, None
for repo, dcol, scol in CANDIDATES:
    try:
        d = load_dataset(repo)
        cols = d["train"].column_names
        if dcol not in cols:
            continue
        if scol not in cols:                       # some mirrors ship summary1/2/3
            alt = [c for c in cols if c.startswith("summary")]
            if not alt:
                continue
            scol = alt[0]
        raw, DATASET_REPO, DIA_COL, SUM_COL = d, repo, dcol, scol
        print(f"Loaded {repo}  dialogue='{dcol}'  summary='{scol}'")
        break
    except Exception as e:
        print(f"  {repo} unavailable ({type(e).__name__})")
if raw is None:
    raise RuntimeError("No summarization dataset could be loaded.")

def clean(ex):
    return {"dialogue": (ex[DIA_COL] or "").strip(),
            "summary":  (ex[SUM_COL] or "").strip()}

ds = raw.map(clean, remove_columns=[c for c in raw["train"].column_names
                                     if c not in ("dialogue", "summary")])
ds = ds.filter(lambda x: len(x["dialogue"]) > 50 and len(x["summary"]) > 10)

if "validation" not in ds:
    sp = ds["train"].train_test_split(test_size=0.05, seed=SEED)
    ds["train"], ds["validation"] = sp["train"], sp["test"]

if MAX_TRAIN_SAMPLES:
    ds["train"] = ds["train"].shuffle(seed=SEED).select(
        range(min(MAX_TRAIN_SAMPLES, len(ds["train"]))))

eval_split = "test" if "test" in ds else "validation"
eval_set = ds[eval_split].shuffle(seed=SEED).select(
    range(min(EVAL_SAMPLES, len(ds[eval_split]))))

RESULTS["dataset_repo"] = DATASET_REPO
RESULTS["splits"] = {k: len(v) for k, v in ds.items()}
RESULTS["eval_split"] = eval_split
RESULTS["eval_n"] = len(eval_set)
print(RESULTS["splits"], "| eval on", len(eval_set), "from", eval_split)

ex = ds["train"][0]
print("\n--- example dialogue ---")
print(textwrap.fill(ex["dialogue"][:600], 100))
print("--- reference summary ---")
print(textwrap.fill(ex["summary"], 100))

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

{'train': 6000, 'validation': 500, 'test': 1500} | eval on 250 from test

--- example dialogue ---
#Person1#: Hello, Anna speaking! #Person2#: Hey, Anna, this is Jason. #Person1#: Jason, where have
you been hiding lately? You know it's been a long time since your last call. Have you been good?
#Person2#: Yes. How are you, Anna? #Person1#: I am fine. What have you been doing? #Person2#:
Working. I've been really busy these days. I got a promotion. #Person1#: That's great,
congratulations! #Person2#: Thanks. I am feeling pretty good about myself too. You know, bigger
office, a raise and even an assistant. #Person1#: That's good. So I guess I'll have to make an
appointment to see you. #Person2
--- reference summary ---
Jason hasn't called Anna for a long time. He calls her to tell her he got a promotion and he feels
good about it. Anna invites him to come over to her house tonight to get drunk.


## 2. Prompt format and completion-only loss

Two details that separate a working SFT run from one that looks fine and generalises badly:

**Use the model's own chat template.** Qwen2.5-Instruct was aligned with a specific special-token layout. Inventing a prompt format discards that alignment and makes the zero-shot baseline incomparable, since the base model would be judged on a format it was never trained for.

**Mask the prompt out of the loss.** By default a causal LM is trained to predict every token, including the instruction and the input dialogue. The model then spends most of its gradient learning to reproduce text it will always be *given*. Setting those positions to `-100` restricts the loss to the summary — the only part the model actually has to produce.

In [5]:
# ============================================================
# 4. Tokenizer, prompt construction, label masking
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM = "You summarise conversations."
INSTRUCTION = ("Summarise the conversation below in two or three sentences. "
               "Write in the third person, name the speakers, and state only what "
               "the conversation says.")

def build_prompt(dialogue, budget=560):
    """Chat-template prompt, with the dialogue truncated to a token budget so the
    instruction and the summary are never the thing that gets cut off."""
    ids = tokenizer(dialogue, add_special_tokens=False).input_ids
    if len(ids) > budget:
        dialogue = tokenizer.decode(ids[:budget], skip_special_tokens=True) + " ..."
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"{INSTRUCTION}\n\n{dialogue}"}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode(ex):
    prompt_ids = tokenizer(build_prompt(ex["dialogue"]), add_special_tokens=False).input_ids
    answer_ids = tokenizer(ex["summary"], add_special_tokens=False).input_ids
    answer_ids = answer_ids + [tokenizer.eos_token_id]      # teach it to stop
    input_ids  = (prompt_ids + answer_ids)[:MAX_LENGTH]
    labels     = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LENGTH]
    return {"input_ids": input_ids, "labels": labels,
            "attention_mask": [1] * len(input_ids)}

train_tok = ds["train"].map(encode, remove_columns=ds["train"].column_names)
val_tok   = ds["validation"].map(encode, remove_columns=ds["validation"].column_names)

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:                                   # right-pad for training
        n = maxlen - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * n)
        out["attention_mask"].append(b["attention_mask"] + [0] * n)
        out["labels"].append(b["labels"] + [-100] * n)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

lens = [len(x) for x in train_tok["input_ids"]]
sup  = [sum(1 for t in l if t != -100) for l in train_tok["labels"][:500]]
RESULTS["token_len"] = {"median": int(np.median(lens)), "p95": int(np.percentile(lens, 95)),
                        "max": int(max(lens)), "median_supervised": int(np.median(sup))}
print("sequence length  median={median}  p95={p95}  max={max}".format(**RESULTS["token_len"]))
print(f"supervised tokens per example (median): {RESULTS['token_len']['median_supervised']}")
print(f"-> loss is computed on ~{RESULTS['token_len']['median_supervised']/np.median(lens):.0%} "
      f"of each sequence; the rest is the prompt and is masked out")

# Sanity check: decode exactly what the model is scored on.
s = train_tok[0]
print("\n--- supervised span ---")
print(tokenizer.decode([t for t in s["labels"] if t != -100]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

sequence length  median=254  p95=464  max=768
supervised tokens per example (median): 32
-> loss is computed on ~13% of each sequence; the rest is the prompt and is masked out

--- supervised span ---
Jason hasn't called Anna for a long time. He calls her to tell her he got a promotion and he feels good about it. Anna invites him to come over to her house tonight to get drunk.<|im_end|>


## 3. Baselines before training

Running these first is not a formality. On summarization, two things are routinely true and routinely unreported:

- Lead-3 beats a lot of trained models on ROUGE
- An instruction-tuned base model, correctly prompted, is already decent

If the fine-tuned adapter does not clear both, the honest conclusion is that the run did not help — and that conclusion is much harder to reach after you have already written the model card.

In [6]:
# ============================================================
# 5. ROUGE scorer and the lead-3 baseline
# ============================================================
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeLsum"], use_stemmer=True)

def rouge(preds, refs):
    agg = {"rouge1": [], "rouge2": [], "rougeLsum": []}
    for p, r in zip(preds, refs):
        p_ = "\n".join(re.split(r"(?<=[.!?]) +", p.strip())) or " "
        r_ = "\n".join(re.split(r"(?<=[.!?]) +", r.strip())) or " "
        sc = scorer.score(r_, p_)
        for k in agg:
            agg[k].append(sc[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in agg.items()}

def length_stats(preds):
    w = [len(p.split()) for p in preds]
    return {"mean_words": float(np.mean(w)), "median_words": float(np.median(w))}

eval_dialogues = list(eval_set["dialogue"])
eval_refs      = list(eval_set["summary"])

lead3 = [" ".join(d.strip().split("\n")[:3]) for d in eval_dialogues]
RESULTS["baseline_lead3"] = {"name": "Lead-3 (first three turns, verbatim)",
                             **rouge(lead3, eval_refs), **length_stats(lead3)}
RESULTS["reference_length"] = length_stats(eval_refs)
print(json.dumps(RESULTS["baseline_lead3"], indent=2))
print(f"\nreference length: {RESULTS['reference_length']}")

{
  "name": "Lead-3 (first three turns, verbatim)",
  "rouge1": 0.26577740697497126,
  "rouge2": 0.06464371648910054,
  "rougeLsum": 0.24603533403187972,
  "mean_words": 39.316,
  "median_words": 34.0
}

reference length: {'mean_words': 19.52, 'median_words': 18.0}


## 4. Load the base model and measure it before touching it

The base model is loaded once in fp16 and used twice: first to produce the zero-shot baseline, then as the frozen backbone the adapter attaches to. Same weights, same prompt, same decoding settings — which is what makes the before/after comparison mean anything.

Greedy decoding throughout. Sampling would make both numbers noisy and the comparison unreproducible.

In [7]:
# ============================================================
# 6. Load base model (fp16, no quantization)
# ============================================================
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    attn_implementation=ATTN_IMPL,
    device_map={"": 0} if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
print(f"loaded in {time.time()-t0:.0f}s")
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

n_base = sum(p.numel() for p in model.parameters())
RESULTS["base_params"] = int(n_base)
print(f"base parameters: {n_base/1e9:.2f}B")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded in 31s
GPU allocated: 3.09 GB
base parameters: 1.54B


In [8]:
# ============================================================
# 7. Batched greedy generation helper
#
# Left padding for decoder-only generation: right padding would place pad
# tokens between the prompt and the first generated token, so the model would
# continue from padding instead of from the prompt.
# ============================================================
@torch.no_grad()
def generate(dialogues, batch_size=GEN_BATCH, max_new_tokens=MAX_NEW_TOKENS, desc=""):
    model.eval()
    model.config.use_cache = True
    old_side, tokenizer.padding_side = tokenizer.padding_side, "left"
    out, t_start = [], time.time()
    for i in range(0, len(dialogues), batch_size):
        chunk = dialogues[i:i + batch_size]
        enc = tokenizer([build_prompt(d) for d in chunk], return_tensors="pt",
                        padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             num_beams=1, pad_token_id=tokenizer.pad_token_id)
        for j in range(len(chunk)):
            new = gen[j][enc["input_ids"].shape[1]:]
            out.append(tokenizer.decode(new, skip_special_tokens=True).strip())
        done = min(i + batch_size, len(dialogues))
        if done % (batch_size * 5) == 0 or done == len(dialogues):
            el = time.time() - t_start
            print(f"  {desc} {done}/{len(dialogues)}  {el:.0f}s elapsed, "
                  f"~{el/done*(len(dialogues)-done):.0f}s left", end="\r")
    tokenizer.padding_side = old_side
    print()
    return out

In [9]:
# ============================================================
# 8. Baseline 2 — zero-shot base model
#
# This is the number that decides whether the fine-tune was worth running.
# ============================================================
zs_preds = generate(eval_dialogues, desc="zero-shot")
RESULTS["baseline_zeroshot"] = {"name": f"{BASE_MODEL} zero-shot (same prompt)",
                                **rouge(zs_preds, eval_refs), **length_stats(zs_preds)}
print(json.dumps(RESULTS["baseline_zeroshot"], indent=2))

print("\n--- sample zero-shot output ---")
print(textwrap.fill(zs_preds[0], 100))
print("--- reference ---")
print(textwrap.fill(eval_refs[0], 100))

  zero-shot 250/250  130s elapsed, ~0s left
{
  "name": "Qwen/Qwen2.5-1.5B-Instruct zero-shot (same prompt)",
  "rouge1": 0.2822324253093497,
  "rouge2": 0.07561242769176264,
  "rougeLsum": 0.2423744197235219,
  "mean_words": 49.356,
  "median_words": 48.0
}

--- sample zero-shot output ---
The conversation discusses Islam, specifically the Hajj pilgrimage to Mecca by Muslim men. People
visit for religious reasons but also because it can lead to health issues due to overcrowding and
accidents. The speaker mentions other places like Lourdes where some believe healing occurs through
faith.
--- reference ---
#Person1# and #Person2# talk about pilgrims around the world, including Muslims' pilgrimage to mecca
and Christians' pilgrimage to Canterbury or Vatican. #Person2# thinks faith heals people instead of
magical places.


## 5. Attach the LoRA adapter

Two things in the cell below are easy to get wrong and silent when you do.

**Cast the trainable parameters to fp32.** The base is loaded in fp16, so LoRA's `A` and `B` matrices inherit fp16. Training with `fp16=True` then fails with *"Attempting to unscale FP16 gradients"* — the gradient scaler needs fp32 master weights. Casting only the adapter back to fp32 costs a few megabytes and nothing else. (QLoRA users get this for free from `prepare_model_for_kbit_training`, which is why the error surprises people moving from QLoRA to plain LoRA.)

**Disable the KV cache during training.** `use_cache=True` conflicts with gradient checkpointing; transformers will warn and silently turn checkpointing off, and the run then OOMs for no visible reason.

On rank: `r=16` with `alpha=32` gives a scaling of 2.0. The usual rule of thumb is `alpha = 2r`. Raising `r` without raising `alpha` effectively shrinks the update.

In [11]:
# ============================================================
# 9. LoRA configuration
# ============================================================

# --- compatibility shim: peft vs Colab's preinstalled torchao -------------
# peft's is_torchao_available() RAISES ImportError when it finds a torchao
# older than its minimum, instead of returning False. Colab ships torchao
# 0.10.0; current peft wants >= 0.16. Nothing in this notebook uses torchao,
# so the honest answer to "is it available" is just False.
import sys
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available()
    print("torchao check: OK")
except ImportError as _e:
    print("patching peft torchao check ->", _e)
    def _no_torchao():
        return False
    _piu.is_torchao_available = _no_torchao
    # Other peft modules did `from .import_utils import is_torchao_available`,
    # so they hold their own reference. Rebind every copy.
    _n = 0
    for _m in list(sys.modules.values()):
        if (_m is not None
                and getattr(_m, "__name__", "").startswith("peft")
                and hasattr(_m, "is_torchao_available")):
            _m.is_torchao_available = _no_torchao
            _n += 1
    print(f"rebound in {_n} peft modules")
except Exception as _e:
    print("torchao check raised something unexpected:", type(_e).__name__, _e)
# --------------------------------------------------------------------------

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)

# fp16 base + fp16 adapter + GradScaler = "Attempting to unscale FP16 gradients".
# Keep the trainable parameters in fp32; the frozen base stays fp16.
if DTYPE == torch.float16:
    for _, p in model.named_parameters():
        if p.requires_grad:
            p.data = p.data.float()

model.config.use_cache = False          # conflicts with gradient checkpointing
model.enable_input_require_grads()      # needed for checkpointing with a frozen base

model.print_trainable_parameters()
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
RESULTS["lora"] = {
    "r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "trainable_params": int(n_train),
    "trainable_pct": float(100 * n_train / n_base),
    "adapter_mb": float(n_train * 2 / 1e6),
}
print(f"\ntrainable: {n_train/1e6:.2f}M ({RESULTS['lora']['trainable_pct']:.2f}% of base)")
print(f"adapter on disk (fp16): ~{RESULTS['lora']['adapter_mb']:.0f} MB "
      f"vs ~{n_base*2/1e9:.1f} GB for a full copy")

patching peft torchao check -> Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported
rebound in 4 peft modules
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

trainable: 18.46M (1.20% of base)
adapter on disk (fp16): ~37 MB vs ~3.1 GB for a full copy


## 6. Training

- **2e-4**, ten times the full-fine-tuning rate. Far fewer parameters have to carry the same update, so each one has to move further.
- **Cosine schedule with 5% warmup.** Shorter warmup than an encoder run needs, because there is no randomly-initialised head here — the adapter starts at zero by construction (`B` is zero-initialised, so the model begins exactly as the base model).
- **Effective batch 16** via 2 × 8 accumulation. The T4 cannot hold 16 sequences of 768 tokens at once; accumulation gets the same gradient at the cost of wall-clock time.
- **Gradient checkpointing** trades about 30% speed for roughly half the activation memory.
- Selection on **eval loss**, which is a proxy. The metric that matters is ROUGE against the baselines, computed afterwards — loss can improve while generations get worse (longer, more repetitive). That gap is the reason the evaluation section exists.

In [12]:
# ============================================================
# 10. Training arguments — version-robust
#
# TrainingArguments has been renamed and pruned across releases. Inspect what
# this installed version accepts, pass only that, and report anything dropped
# instead of failing on a keyword.
# ============================================================
import transformers
print("transformers:", transformers.__version__, "| peft:", __import__("peft").__version__)

supported = set(inspect.signature(TrainingArguments.__init__).parameters)
supported |= set(getattr(TrainingArguments, "__dataclass_fields__", {}))
supported -= {"self", "kwargs"}

steps_per_epoch = math.ceil(len(train_tok) / (MICRO_BATCH * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))

wanted = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(DTYPE == torch.float16),
    bf16=(DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    optim="adamw_torch",
    max_grad_norm=1.0,
    seed=SEED,
    report_to="none",
)

if "eval_strategy" in supported:
    wanted["eval_strategy"] = "epoch"
elif "evaluation_strategy" in supported:
    wanted["evaluation_strategy"] = "epoch"

manual_schedule = False
if "warmup_ratio" in supported:
    wanted["warmup_ratio"] = WARMUP_RATIO
elif "warmup_steps" in supported:
    wanted["warmup_steps"] = warmup_steps
else:
    manual_schedule = True
    print("WARNING: no warmup argument available; building the scheduler by hand.")

if "gradient_checkpointing_kwargs" in supported:
    wanted["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

kw = {k: v for k, v in wanted.items() if k in supported}
dropped = sorted(set(wanted) - set(kw))
if dropped:
    print("Dropped (unsupported by this version):", dropped)

training_args = TrainingArguments(**kw)

optimizers = (None, None)
if manual_schedule:
    from torch.optim import AdamW
    from transformers import get_cosine_schedule_with_warmup
    params = [p for p in model.parameters() if p.requires_grad]
    opt = AdamW(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    optimizers = (opt, get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps))

tp = set(inspect.signature(Trainer.__init__).parameters)
trainer_kw = dict(model=model, args=training_args, train_dataset=train_tok,
                  eval_dataset=val_tok, data_collator=collate)
if "processing_class" in tp:
    trainer_kw["processing_class"] = tokenizer
elif "tokenizer" in tp:
    trainer_kw["tokenizer"] = tokenizer
if manual_schedule:
    trainer_kw["optimizers"] = optimizers

trainer = Trainer(**trainer_kw)
print(f"steps/epoch={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}  "
      f"effective batch={MICRO_BATCH * GRAD_ACCUM}")

transformers: 5.19.0 | peft: 0.21.2
steps/epoch=375  total=750  warmup=37  effective batch=16


In [13]:
# ============================================================
# 11. Train
# ============================================================
torch.cuda.reset_peak_memory_stats()
train_out = trainer.train()

RESULTS["train_runtime_sec"] = float(train_out.metrics.get("train_runtime", 0.0))
RESULTS["train_loss"] = float(train_out.metrics.get("train_loss", float("nan")))
RESULTS["total_steps"] = total_steps
RESULTS["peak_gpu_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)

hist = [h for h in trainer.state.log_history if "eval_loss" in h]
RESULTS["eval_loss_by_epoch"] = [{"epoch": h.get("epoch"), "eval_loss": h["eval_loss"]}
                                 for h in hist]
print(f"\ntraining time: {RESULTS['train_runtime_sec']/60:.1f} min")
print(f"peak GPU memory: {RESULTS['peak_gpu_gb']:.2f} GB")
for h in RESULTS["eval_loss_by_epoch"]:
    print(f"  epoch {h['epoch']:.0f}: eval_loss {h['eval_loss']:.4f}")

Epoch,Training Loss,Validation Loss
1,0.942884,0.923045
2,0.784348,0.927386



training time: 62.8 min
peak GPU memory: 6.35 GB
  epoch 1: eval_loss 0.9230
  epoch 2: eval_loss 0.9274


## 7. Evaluation

Same evaluation set, same prompt, same greedy decoding as the zero-shot run. The only thing that changed is the adapter.

Beyond ROUGE, three checks that catch the failure modes ROUGE is blind to:

- **Length.** A model that learns to produce reference-length output gains ROUGE without understanding anything. Comparing predicted length to reference length tells you how much of the gain is that.
- **Extractive copying.** The fraction of summaries that lift a long span verbatim from the dialogue. High copying means the model found the shortcut rather than the task.
- **Degeneration.** Empty outputs and repeated text, which ROUGE scores as merely mediocre rather than broken.

In [14]:
# ============================================================
# 12. Generate with the adapter and score
# ============================================================
model.config.use_cache = True
ft_preds = generate(eval_dialogues, desc="fine-tuned")

RESULTS["model"] = {"name": "LoRA fine-tuned",
                    **rouge(ft_preds, eval_refs), **length_stats(ft_preds)}

def copy_rate(preds, sources, n=12):
    """Fraction of predictions containing an n-word span lifted verbatim."""
    hits = 0
    for p, s in zip(preds, sources):
        pw, sl = p.split(), " " + " ".join(s.split()).lower() + " "
        if any(" " + " ".join(pw[i:i + n]).lower() + " " in sl
               for i in range(max(0, len(pw) - n + 1))):
            hits += 1
    return hits / max(1, len(preds))

def repeat_rate(preds):
    out = 0
    for p in preds:
        w = p.lower().split()
        if len(w) > 8 and len(set(w)) / len(w) < 0.5:
            out += 1
    return out / max(1, len(preds))

for tag, preds in (("baseline_zeroshot", zs_preds), ("model", ft_preds)):
    RESULTS[tag]["empty_rate"] = float(np.mean([len(p.strip()) == 0 for p in preds]))
    RESULTS[tag]["copy_rate"] = float(copy_rate(preds, eval_dialogues))
    RESULTS[tag]["repeat_rate"] = float(repeat_rate(preds))

print(json.dumps(RESULTS["model"], indent=2))

  fine-tuned 250/250  167s elapsed, ~0s left
{
  "name": "LoRA fine-tuned",
  "rouge1": 0.41845302150901786,
  "rouge2": 0.15212679954051714,
  "rougeLsum": 0.3623705738893997,
  "mean_words": 24.588,
  "median_words": 24.0,
  "empty_rate": 0.0,
  "copy_rate": 0.016,
  "repeat_rate": 0.0
}


In [15]:
# ============================================================
# 13. The comparison table
# ============================================================
rows = [RESULTS["baseline_lead3"], RESULTS["baseline_zeroshot"], RESULTS["model"]]
df = pd.DataFrame([{
    "system": r["name"],
    "ROUGE-1": round(r["rouge1"], 4),
    "ROUGE-2": round(r["rouge2"], 4),
    "ROUGE-Lsum": round(r["rougeLsum"], 4),
    "mean words": round(r["mean_words"], 1),
} for r in rows])
print(df.to_string(index=False))
print(f"\nreference mean words: {RESULTS['reference_length']['mean_words']:.1f}")

zs, ft = RESULTS["baseline_zeroshot"], RESULTS["model"]
delta = {k: ft[k] - zs[k] for k in ("rouge1", "rouge2", "rougeLsum")}
RESULTS["delta_vs_zeroshot"] = delta
RESULTS["beats_zeroshot"] = bool(delta["rougeLsum"] > 0)
RESULTS["beats_lead3"] = bool(ft["rougeLsum"] > RESULTS["baseline_lead3"]["rougeLsum"])

print(f"\nvs zero-shot: R1 {delta['rouge1']:+.4f}  R2 {delta['rouge2']:+.4f}  "
      f"RLsum {delta['rougeLsum']:+.4f}")
print("beats zero-shot:", RESULTS["beats_zeroshot"], "| beats lead-3:", RESULTS["beats_lead3"])
if not RESULTS["beats_zeroshot"]:
    print("\nThe adapter did not beat prompting the base model. That is a real result, "
          "not a bug to hide: report it, and treat r / lr / epochs as the next thing "
          "to vary rather than the model card as the next thing to write.")

print("\nbehavioural checks:")
print(f"  {'':<14}{'copy':<10}{'repeat':<10}{'empty'}")
for tag, label in (("baseline_zeroshot", "zero-shot"), ("model", "fine-tuned")):
    r = RESULTS[tag]
    print(f"  {label:<14}{r['copy_rate']:<10.3f}{r['repeat_rate']:<10.3f}{r['empty_rate']:.3f}")

# Figure for the model card
fig, ax = plt.subplots(figsize=(5.4, 3.0), dpi=130)
labels = ["Lead-3", "Zero-shot", "LoRA"]
x = np.arange(3); w = 0.26
for i, m in enumerate(["rouge1", "rouge2", "rougeLsum"]):
    ax.bar(x + (i - 1) * w, [r[m] for r in rows], w, label=m.upper())
ax.set_xticks(x, labels); ax.set_ylabel("F-measure"); ax.legend(frameon=False)
ax.set_title("ROUGE against baselines"); ax.grid(axis="y", alpha=.25)
fig.tight_layout(); fig.savefig(os.path.join(ADAPTER_DIR, "rouge_comparison.png"))
plt.close(fig)

                                            system  ROUGE-1  ROUGE-2  ROUGE-Lsum  mean words
              Lead-3 (first three turns, verbatim)   0.2658   0.0646      0.2460        39.3
Qwen/Qwen2.5-1.5B-Instruct zero-shot (same prompt)   0.2822   0.0756      0.2424        49.4
                                   LoRA fine-tuned   0.4185   0.1521      0.3624        24.6

reference mean words: 19.5

vs zero-shot: R1 +0.1362  R2 +0.0765  RLsum +0.1200
beats zero-shot: True | beats lead-3: True

behavioural checks:
                copy      repeat    empty
  zero-shot     0.000     0.000     0.000
  fine-tuned    0.016     0.000     0.000


In [16]:
# ============================================================
# 14. Qualitative side-by-side
#
# Numbers tell you whether something changed. Reading output tells you what.
# ============================================================
samples = []
for i in range(min(5, len(eval_refs))):
    samples.append({"dialogue": eval_dialogues[i][:400],
                    "reference": eval_refs[i],
                    "zero_shot": zs_preds[i],
                    "fine_tuned": ft_preds[i]})
    print("=" * 100)
    print("DIALOGUE   ", textwrap.shorten(eval_dialogues[i].replace("\n", " / "), 260))
    print("REFERENCE  ", textwrap.fill(eval_refs[i], 96, subsequent_indent=" " * 12))
    print("ZERO-SHOT  ", textwrap.fill(zs_preds[i], 96, subsequent_indent=" " * 12))
    print("FINE-TUNED ", textwrap.fill(ft_preds[i], 96, subsequent_indent=" " * 12))
RESULTS["samples"] = samples

DIALOGUE    #Person1#: Hi! What are you watching? / #Person2#: It's a program about islam. It's very interesting. / #Person1#: Wow! So many people! Where are they and what are they doing? / #Person2#: They are muslims on a pilgrimage to mecca. Muslims call this [...]
REFERENCE   #Person1# and #Person2# talk about pilgrims around the world, including Muslims' pilgrimage to
            mecca and Christians' pilgrimage to Canterbury or Vatican. #Person2# thinks faith
            heals people instead of magical places.
ZERO-SHOT   The conversation discusses Islam, specifically the Hajj pilgrimage to Mecca by Muslim men.
            People visit for religious reasons but also because it can lead to health issues due
            to overcrowding and accidents. The speaker mentions other places like Lourdes where
            some believe healing occurs through faith.
FINE-TUNED  #Person2# tells #Person1# it's a program about Islam. #Person2# explains that Muslims believe
            that every

In [18]:
# Diagnostic A: strip zero-shot down to one sentence, rescore
zs_first = [re.split(r"(?<=[.!?]) ", p.strip())[0] for p in zs_preds]
print("zero-shot, first sentence only:", rouge(zs_first, eval_refs), length_stats(zs_first))
print("zero-shot, full:              ", {k: round(RESULTS['baseline_zeroshot'][k], 4)
                                          for k in ('rouge1','rouge2','rougeLsum')})
print("LoRA:                         ", {k: round(RESULTS['model'][k], 4)
                                          for k in ('rouge1','rouge2','rougeLsum')})

zero-shot, first sentence only: {'rouge1': 0.2792096684665398, 'rouge2': 0.07028166198280064, 'rougeLsum': 0.2360648080062642} {'mean_words': 16.6, 'median_words': 15.0}
zero-shot, full:               {'rouge1': 0.2822, 'rouge2': 0.0756, 'rougeLsum': 0.2424}
LoRA:                          {'rouge1': 0.4185, 'rouge2': 0.1521, 'rougeLsum': 0.3624}


In [19]:
# Diagnostic B: length-matched zero-shot baseline (adapter disabled)
_orig = INSTRUCTION
INSTRUCTION = ("Summarise the conversation below in ONE or TWO sentences, under 25 "
               "words total. Write in the third person and name the speakers.")
try:
    with model.disable_adapter():
        zs_short = generate(eval_dialogues, desc="zero-shot short")
finally:
    INSTRUCTION = _orig

RESULTS["baseline_zeroshot_lengthmatched"] = {
    "name": "Zero-shot, length-matched prompt", **rouge(zs_short, eval_refs),
    **length_stats(zs_short)}
print(json.dumps(RESULTS["baseline_zeroshot_lengthmatched"], indent=2))

gap = RESULTS["model"]["rouge1"] - RESULTS["baseline_zeroshot_lengthmatched"]["rouge1"]
print(f"\nR1 gap vs length-matched zero-shot: {gap:+.4f}  "
      f"(vs {RESULTS['delta_vs_zeroshot']['rouge1']:+.4f} against the original prompt)")

  zero-shot short 250/250  93s elapsed, ~0s left
{
  "name": "Zero-shot, length-matched prompt",
  "rouge1": 0.30137998875976074,
  "rouge2": 0.07960654737549992,
  "rougeLsum": 0.2502457133674181,
  "mean_words": 27.904,
  "median_words": 25.5
}

R1 gap vs length-matched zero-shot: +0.1171  (vs +0.1362 against the original prompt)


In [20]:
# ============================================================
# 15. Save the adapter
#
# ~70MB instead of ~3GB. The adapter is useless without the base model, so the
# base model id has to be recorded — adapter_config.json stores it, which is
# why the config must be saved alongside the weights.
# ============================================================
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

with open(os.path.join(ADAPTER_DIR, "metrics.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)

size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
              for f in os.listdir(ADAPTER_DIR)
              if os.path.isfile(os.path.join(ADAPTER_DIR, f))) / 1e6
RESULTS["adapter_dir_mb"] = float(size_mb)
print(f"adapter directory: {size_mb:.1f} MB")
print(sorted(os.listdir(ADAPTER_DIR)))

adapter directory: 85.4 MB
['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'metrics.json', 'rouge_comparison.png', 'tokenizer.json', 'tokenizer_config.json']


## 8. The model card

Generated from `RESULTS`, so every number in it was measured in this run. A card with hardcoded metrics describes whatever the model was the day the card was written, and nobody notices when that stops being true.

For an adapter repository the card carries one extra obligation: it has to name the exact base model and show how to load the two together, because the weights in the repo are meaningless on their own.

In [24]:
# ============================================================
# 16. Generate README.md (model card)
#
# Every number comes from RESULTS, so the card describes this run and not a
# remembered one. Re-run this cell after any new measurement.
# ============================================================
m, zs_b, l3 = RESULTS["model"], RESULTS["baseline_zeroshot"], RESULTS["baseline_lead3"]
lr_ = RESULTS["lora"]
d = RESULTS["delta_vs_zeroshot"]

verdict = ("It improves on prompting the base model."
           if RESULTS["beats_zeroshot"] else
           "**It does not improve on prompting the base model.** The adapter is published "
           "for reproducibility, not because it is the better option for this task.")

sample_block = "\n\n".join(
    f"**Reference**\n> {s['reference']}\n\n**Zero-shot**\n> {s['zero_shot']}\n\n"
    f"**LoRA**\n> {s['fine_tuned']}"
    for s in RESULTS["samples"][:3])

# ---- Length-controlled attribution (only if the diagnostic has been run) ----
length_section = ""
if "baseline_zeroshot_lengthmatched" in RESULTS:
    lm = RESULTS["baseline_zeroshot_lengthmatched"]

    attr = {}
    for k in ("rouge1", "rouge2", "rougeLsum"):
        total = m[k] - zs_b[k]
        from_len = lm[k] - zs_b[k]
        attr[k] = {"total_gain": float(total), "from_length": float(from_len),
                   "length_share": float(from_len / total) if total else 0.0}
    RESULTS["length_attribution"] = attr

    attr_rows = "\n".join(
        f"| {k.upper()} | {zs_b[k]:.4f} | {lm[k]:.4f} | {m[k]:.4f} | "
        f"{attr[k]['length_share']:.0%} |"
        for k in ("rouge1", "rouge2", "rougeLsum"))

    fs = RESULTS.get("baseline_zeroshot_firstsent")
    first_sent_note = ""
    if fs:
        first_sent_note = (
            f"\nTruncating zero-shot output to its first sentence ({fs['mean_words']:.1f} "
            f"words) leaves ROUGE-1 essentially unchanged ({fs['rouge1']:.4f} vs "
            f"{zs_b['rouge1']:.4f}). The base model's weakness is *what* it selects, not "
            f"how much it writes.\n")

    length_section = f"""### Is the gain just length?

Fine-tuned summaries are shorter than zero-shot ones ({m['mean_words']:.1f} vs
{zs_b['mean_words']:.1f} words, against a reference mean of
{RESULTS['reference_length']['mean_words']:.1f}), and ROUGE F-measure rewards matching the
reference length. To separate that effect from real improvement, the base model was re-run
with a prompt asking for under 25 words, producing {lm['mean_words']:.1f}-word summaries —
close to the adapter's own length.

| Metric | Zero-shot | Zero-shot, length-matched | This adapter | share of gain from length |
|---|---|---|---|---|
{attr_rows}

Matching the length recovers only a small part of the gap. ROUGE-2 is the clearest case:
bigram overlap cannot be bought with length, and the adapter still scores roughly double
the length-matched baseline ({m['rouge2']:.4f} vs {lm['rouge2']:.4f}).
{first_sent_note}
"""

card = f"""---
language:
- en
license: apache-2.0
library_name: peft
pipeline_tag: summarization
base_model: {BASE_MODEL}
datasets:
- {RESULTS["dataset_repo"]}
tags:
- lora
- peft
- summarization
- dialogue-summarization
- qwen2
metrics:
- rouge
model-index:
- name: {REPO_NAME}
  results:
  - task:
      type: summarization
      name: Dialogue Summarization
    dataset:
      type: {RESULTS["dataset_repo"]}
      name: DialogSum
      split: {RESULTS["eval_split"]}
    metrics:
    - type: rouge
      value: {m["rouge1"]:.4f}
      name: ROUGE-1
    - type: rouge
      value: {m["rouge2"]:.4f}
      name: ROUGE-2
    - type: rouge
      value: {m["rougeLsum"]:.4f}
      name: ROUGE-Lsum
---

# {REPO_NAME}

LoRA adapter for [`{BASE_MODEL}`](https://huggingface.co/{BASE_MODEL}), trained on
[{RESULTS["dataset_repo"]}](https://huggingface.co/datasets/{RESULTS["dataset_repo"]}) for
abstractive dialogue summarization.

This repository contains **adapter weights only** ({RESULTS["adapter_dir_mb"]:.0f} MB). It
does nothing without the base model.

| | |
|---|---|
| Trainable parameters | {lr_["trainable_params"]/1e6:.1f}M ({lr_["trainable_pct"]:.2f}% of the base) |
| Training examples | {RESULTS["splits"]["train"]:,} |
| Training time | {RESULTS["train_runtime_sec"]/60:.0f} min on one T4 |
| Peak GPU memory | {RESULTS["peak_gpu_gb"]:.1f} GB |

## Usage

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    "{BASE_MODEL}", torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "{REPO_ID}")
tok = AutoTokenizer.from_pretrained("{REPO_ID}")

messages = [
    dict(role="system", content="{SYSTEM}"),
    dict(role="user", content="{INSTRUCTION}" + "\\n\\n" + your_dialogue),
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**ids, max_new_tokens={MAX_NEW_TOKENS}, do_sample=False)
print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
```

The prompt format matters. The adapter was trained behind this exact chat template and
instruction string; a different wrapper will degrade it.

## Evaluation

{RESULTS["eval_n"]} held-out examples from the `{RESULTS["eval_split"]}` split. Greedy
decoding, `max_new_tokens={MAX_NEW_TOKENS}`, identical prompt for every system.

| System | ROUGE-1 | ROUGE-2 | ROUGE-Lsum | mean words |
|---|---|---|---|---|
| {l3["name"]} | {l3["rouge1"]:.4f} | {l3["rouge2"]:.4f} | {l3["rougeLsum"]:.4f} | {l3["mean_words"]:.1f} |
| {zs_b["name"]} | {zs_b["rouge1"]:.4f} | {zs_b["rouge2"]:.4f} | {zs_b["rougeLsum"]:.4f} | {zs_b["mean_words"]:.1f} |
| **This adapter** | **{m["rouge1"]:.4f}** | **{m["rouge2"]:.4f}** | **{m["rougeLsum"]:.4f}** | {m["mean_words"]:.1f} |
| *(reference summaries)* | | | | *{RESULTS["reference_length"]["mean_words"]:.1f}* |

Change against the zero-shot base model: **ROUGE-1 {d["rouge1"]:+.4f}, ROUGE-2
{d["rouge2"]:+.4f}, ROUGE-Lsum {d["rougeLsum"]:+.4f}**. {verdict}

The zero-shot row is the comparison that matters. Beating lead-3 proves very little, since
the base model already does; the question is whether training beat *prompting the same
weights*.

![ROUGE comparison](rouge_comparison.png)

{length_section}### Behavioural checks

ROUGE is insensitive to several ways a summarizer can be bad, so these are measured
separately:

| | extractive copying | repetition | empty |
|---|---|---|---|
| Zero-shot | {zs_b["copy_rate"]:.3f} | {zs_b["repeat_rate"]:.3f} | {zs_b["empty_rate"]:.3f} |
| This adapter | {m["copy_rate"]:.3f} | {m["repeat_rate"]:.3f} | {m["empty_rate"]:.3f} |

*Copying* is the share of summaries containing a 12-word span lifted verbatim from the
dialogue. *Repetition* flags outputs where fewer than half the words are distinct.

### Examples

{sample_block}

## Training

| | |
|---|---|
| Base model | `{BASE_MODEL}` ({RESULTS["base_params"]/1e9:.2f}B parameters, fp16, not quantized) |
| Method | LoRA (PEFT) |
| Rank / alpha / dropout | {lr_["r"]} / {lr_["alpha"]} / {lr_["dropout"]} |
| Target modules | {", ".join(f"`{t}`" for t in lr_["target_modules"])} |
| Trainable parameters | {lr_["trainable_params"]/1e6:.1f}M ({lr_["trainable_pct"]:.2f}%) |
| Learning rate | {LEARNING_RATE} (AdamW, cosine, {WARMUP_RATIO:.0%} warmup) |
| Effective batch | {MICRO_BATCH} x {GRAD_ACCUM} = {MICRO_BATCH*GRAD_ACCUM} |
| Epochs / steps | {EPOCHS} / {RESULTS["total_steps"]} |
| Max sequence length | {MAX_LENGTH} (median {RESULTS["token_len"]["median"]}) |
| Loss | computed on the summary only; prompt tokens masked to -100 |
| Precision | {"fp16" if DTYPE == torch.float16 else "bf16"} |
| Hardware | one NVIDIA T4 (16 GB), peak {RESULTS["peak_gpu_gb"]:.1f} GB |
| Seed | {SEED} |

Three choices worth stating. The learning rate is **{LEARNING_RATE}**, roughly ten times a
full-fine-tuning rate, because far fewer parameters have to carry the same update — LoRA at
2e-5 barely moves. The loss is computed **only on the summary**: training on the prompt
tokens as well would spend most of the gradient learning to reproduce text the model is
always given. And the adapter parameters are held in fp32 while the base stays fp16, which
is what `GradScaler` needs in order to unscale gradients.

## Limitations

- **Faithfulness is not measured.** ROUGE does not check whether a summary is *true* of the
  dialogue. A hallucinated name, time, or amount scores well as long as the wording
  overlaps. Nothing here tests for that, so do not use this where a fabricated detail
  matters.
- **Domain.** Short, everyday two-party conversations in English. Meetings, many-speaker
  transcripts, domain jargon, and other languages are all out of distribution.
- **Input length.** Dialogues are truncated to about 560 tokens before the prompt is built.
  Longer conversations lose their endings, and the summary reflects that.
- **Single run.** One seed, one configuration. `r={lr_["r"]}` was assumed rather than
  chosen, and no seed-variance estimate was made.
- **Prompt-coupled.** Trained behind one chat template and one instruction string. Treat
  those as part of the model.

## Citation

```bibtex
@inproceedings{{chen2021dialogsum,
  title     = {{DialogSum: A Real-Life Scenario Dialogue Summarization Dataset}},
  author    = {{Chen, Yulong and Liu, Yang and Chen, Liang and Zhang, Yue}},
  booktitle = {{Findings of ACL-IJCNLP 2021}},
  year      = {{2021}}
}}

@article{{hu2021lora,
  title   = {{LoRA: Low-Rank Adaptation of Large Language Models}},
  author  = {{Hu, Edward J. and Shen, Yelong and Wallis, Phillip and others}},
  journal = {{arXiv:2106.09685}},
  year    = {{2021}}
}}
```

---

*Card generated from measured results on {date.today().isoformat()}.*
"""

with open(os.path.join(ADAPTER_DIR, "README.md"), "w") as f:
    f.write(card)
with open(os.path.join(ADAPTER_DIR, "metrics.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)

print(f"model card written: {len(card):,} chars")
print("length-attribution section included:", bool(length_section))
print(card[:900])

model card written: 8,784 chars
length-attribution section included: True
---
language:
- en
license: apache-2.0
library_name: peft
pipeline_tag: summarization
base_model: Qwen/Qwen2.5-1.5B-Instruct
datasets:
- knkarthick/dialogsum
tags:
- lora
- peft
- summarization
- dialogue-summarization
- qwen2
metrics:
- rouge
model-index:
- name: QwenSum-LoRA
  results:
  - task:
      type: summarization
      name: Dialogue Summarization
    dataset:
      type: knkarthick/dialogsum
      name: DialogSum
      split: test
    metrics:
    - type: rouge
      value: 0.4185
      name: ROUGE-1
    - type: rouge
      value: 0.1521
      name: ROUGE-2
    - type: rouge
      value: 0.3624
      name: ROUGE-Lsum
---

# QwenSum-LoRA

LoRA adapter for [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), trained on
[knkarthick/dialogsum](https://huggingface.co/datasets/knkarthick/dialogsum) for
abstractive dialogue summarization.

This repositor


In [25]:
# ============================================================
# 17. Push to the Hub
#
# Colab: Settings (key icon, left sidebar) -> add a secret named HF_TOKEN with a
# WRITE token from https://huggingface.co/settings/tokens, and enable notebook access.
# ============================================================
if PUSH_TO_HUB:
    from huggingface_hub import HfApi, create_repo

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    token = token or os.environ.get("HF_TOKEN")
    if not token:
        from huggingface_hub import notebook_login
        notebook_login()

    create_repo(REPO_ID, repo_type="model", exist_ok=True, token=token)
    HfApi().upload_folder(
        folder_path=ADAPTER_DIR, repo_id=REPO_ID, repo_type="model",
        commit_message=(f"LoRA adapter for {BASE_MODEL} on {RESULTS['dataset_repo']} "
                        f"(ROUGE-Lsum {RESULTS['model']['rougeLsum']:.4f}, "
                        f"{RESULTS['delta_vs_zeroshot']['rougeLsum']:+.4f} vs zero-shot)"),
        token=token,
    )
    print(f"https://huggingface.co/{REPO_ID}")

    if PUSH_MERGED:
        # Merging folds B@A into the base weights: one standalone fp16 model, no PEFT
        # dependency at inference, slightly faster, ~3GB to upload.
        merged = model.merge_and_unload()
        merged_dir = os.path.join(OUTPUT_DIR, "merged")
        merged.save_pretrained(merged_dir, safe_serialization=True)
        tokenizer.save_pretrained(merged_dir)
        create_repo(f"{REPO_ID}-merged", repo_type="model", exist_ok=True, token=token)
        HfApi().upload_folder(folder_path=merged_dir, repo_id=f"{REPO_ID}-merged",
                              repo_type="model", token=token,
                              commit_message="Merged fp16 weights")
        print(f"https://huggingface.co/{REPO_ID}-merged")
else:
    print("PUSH_TO_HUB is False — nothing uploaded.")

https://huggingface.co/emonHasan01/QwenSum-LoRA


In [23]:
# ============================================================
# 18. Inference from the saved adapter
# ============================================================
def summarise(dialogue, max_new_tokens=MAX_NEW_TOKENS):
    model.config.use_cache = True
    model.eval()
    enc = tokenizer(build_prompt(dialogue), return_tensors="pt",
                    add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()

demo = """#Person1#: Hi Rafi, did you manage to look at the deployment logs?
#Person2#: I did. The container keeps restarting every forty seconds.
#Person1#: Memory limit?
#Person2#: That was my guess too, but it is dying at startup, before it allocates much.
#Person1#: Can you roll back to Tuesday's image while we look into it?
#Person2#: Already done. I will open an issue with the stack trace this afternoon."""

print(summarise(demo))

Rafi tells #Person1# that the container keeps restarting because of memory limit. Rafi will open an issue with the stack trace.


---

### Where to go next

Three things this notebook measures but does not fix:

1. **Faithfulness.** ROUGE cannot see a hallucinated name or number. An entailment model scoring each summary against its dialogue, or a per-summary check that every named entity appears in the source, would catch what ROUGE rewards.
2. **Rank.** `r=16` was assumed, not chosen. Running `r ∈ {4, 8, 16, 32}` with alpha held at `2r` costs four short runs and turns the number into a decision — and the answer on small datasets is often lower than people expect.
3. **Seed variance.** One run gives one number. Three seeds give you a spread, and the gap against the zero-shot baseline only means something once you know whether it is bigger than that spread.

And then QLoRA, which is the same pipeline with the base model quantized to 4-bit — worth reaching for when the model no longer fits in fp16, which on a T4 starts somewhere around 3B parameters.
